# Importing dependencies

In [1]:
%pip install joblib

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, check_scoring, accuracy_score
import joblib
from sklearn.model_selection import cross_val_score
import matplotlib
import seaborn as sb
%matplotlib

Using matplotlib backend: module://matplotlib_inline.backend_inline


**Data Collection and Analysis**

In [63]:
sys.path.append(os.path.abspath('../data'))
df_path = '../data/diabetes.csv'
df = pd.read_csv(df_path)

In [64]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [65]:
df.shape

(768, 9)

In [66]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [68]:
diabetic_outcome = df["Outcome"]
diabetic_outcome.value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

0--**Non-diabetic**

1--**Diabetic**

In [69]:
df.groupby('Outcome').mean()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
Outcome,,,,,,,,
0,3.298000,109.980000,68.184000,19.664000,68.792000,30.304200,0.429734,31.190000
1,4.865672,141.257463,70.824627,22.164179,100.335821,35.142537,0.550500,37.067164


Separating data and labels

In [70]:
y = df["Outcome"]
y.head()

0    1
1    0
2    1
3    0
4    1
Name: Outcome, dtype: int64

In [71]:
x = df.drop(["Outcome"], axis= 1)
x.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33


Scale data

In [72]:
scaler = StandardScaler()
scaler.fit(x)
standardized_x = scaler.transform(x)
x = standardized_x
standardized_x

array([[ 0.63994726,  0.84832379,  0.14964075, ...,  0.20401277,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575, ..., -0.68442195,
        -0.36506078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, ..., -1.10325546,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 ,  0.00330087,  0.14964075, ..., -0.73518964,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.1597866 , -0.47073225, ..., -0.24020459,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.8730192 ,  0.04624525, ..., -0.20212881,
        -0.47378505, -0.87137393]])

### Exploratory Data Analysis

Train Test Split

In [73]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify= y, random_state= 42)

In [75]:
print(f"Data has {x.shape} rows and columns")
print(f"Test Data has {x_test.shape} rows and columns")
print(f"Training Data has {x_train.shape} rows and columns")

Data has (768, 8) rows and columns
Test Data has (192, 8) rows and columns
Training Data has (576, 8) rows and columns


Model Training

In [76]:
model_SVC = SVC(kernel= "linear")
model_SVC.fit(x_train, y_train)
model_SVC.score(x_test, y_test)

0.7083333333333334

Model Evaluation

In [90]:
model_SVC.score(x_train, y_train)

0.7899305555555556

In [77]:
ypred = model_SVC.predict(x_test)

In [79]:
print(classification_report(y_test, ypred))
print(confusion_matrix(y_test, ypred))

              precision    recall  f1-score   support

           0       0.75      0.82      0.79       125
           1       0.60      0.49      0.54        67

    accuracy                           0.71       192
   macro avg       0.68      0.66      0.66       192
weighted avg       0.70      0.71      0.70       192

[[103  22]
 [ 34  33]]


In [83]:
cross_validated_score = cross_val_score(model_SVC ,x,y, scoring= "accuracy", cv= 10)
print(cross_validated_score)
print(cross_validated_score.mean())

[0.71428571 0.76623377 0.79220779 0.76623377 0.74025974 0.77922078
 0.80519481 0.79220779 0.75       0.81578947]
0.7721633629528367


Trying a different model

In [86]:
model_logi_reg = LogisticRegression()
model_logi_reg.fit(x_train, y_train)
model_logi_reg.score(x_test, y_test)

0.734375

In [91]:
model_logi_reg.score(x_train, y_train)

0.7951388888888888

In [92]:
ypred_2 = model_logi_reg.predict(x_test)
print(classification_report(y_test, ypred_2))
print(confusion_matrix(y_test, ypred_2))

              precision    recall  f1-score   support

           0       0.77      0.85      0.81       125
           1       0.65      0.52      0.58        67

    accuracy                           0.73       192
   macro avg       0.71      0.69      0.69       192
weighted avg       0.73      0.73      0.73       192

[[106  19]
 [ 32  35]]
